# TonER: A Diacritic-Type-Aware Evaluation Framework for Low-Resource Neural Machine Translation

# Abstract (summary) + Guiding Question

Guiding question:

***Is there a difference in the error rates of different types of diacritical marks in translation models?***

Neural machine translation (using a deep-learning model to translate a language) has become the dominant approach for translating Niger-Congo languages. Unlike English, French or Spanish, these languages are low-resource and often lack sufficient data for training models, resulting in worse accuracy overall. While researchers use metrics to evaluate and eventually fix model translations, existing evaluation methods for translation accuracy often exclude, discredit or ignore diacritical mark accuracy, which is important for many African languages. Of the metrics that do evaluate diacritic accuracy, they treat all diacritical marks as a single category. This obscures whether models fail uniformly across marks or struggle with specific types. This paper introduces a novel framework: TonER. Using TonER, we can evaluate models on types of marks to infer patterns based on the two main types of diacritics, tonal and orthographic marks. After applying TonER to two variants of a model (NLLB-200) across three languages, we found that tonal diacritic error rates substantially exceeded orthographic rates overall.

# 1. Introduction and Background

The three low-resource languages examined in this study are Yorùbá, Igbo, and Ewe. These Niger-Congo languages are tonal and encode meaning through two classes of diacritical marks.

1. Orthographic diacritics modify the identity of a single character. For example, Igbo's underdot transforms o into ọ, the English equivalent of an "a" becoming a "u." This mark is meant mainly for writing and reading.

2. Tonal diacritics mark pitch differences that change the same base word into two forms. For example, Yorùbá's ìgbà means "time or season" but igba means "200". This documents how people actually speak.

However, researchers have noted that tonal marks are regularly excluded from text data: Since tonal marks are included to represent *spoken* words, most text that exists online omits these marks. Native speakers writing in these languages online also omit the marks because they already know the context, it is the English equivalent of adding a mark to say that "car" the vehicle was different than "car" the part of a train. Since there are fewer marks in the data, the models aren't as good at producing these marks when translating. The problem is, models also get the surrounding words wrong in general for languages without much textual data. This results in speakers not being able to infer context AND not having marks to denote what that context would be (Ezeani et al. (2017))

This could create 2 big problems:

1. Recent developments have shifted to highly accurate multilingual model systems for translation (Kudugunta et al., 2023). Since these models are inaccurate in general for low-resource languages, work has shifted toward translation with a model and *then* applying specialized models called diacritizers afterward to restore diacritic marks (Orife, 2018). Again, this could introduce a new problem. These diacritizers are only really trained on "target" language and not the english context. Unlike a native speaker or even a translation model that knows the english translation, these diacritizer models insert the marks in *without* english context which is detrimental to the meaning as previously stated.

2. Standard diacritic error metrics (FER, WER, DER) combine all diacritic-related errors into a single number (Zitouni et al., 2006). This hides whether models are failing on more important tonal marks or the more common orthographic marks.

*This is imperative to document and fix.*

Researchers could report that models are accurate while the translations lose the meaning carried by a word. When researchers eventually use this bad text, they could actually be confusing native speakers and even worse, new learners. Given that these languages are already endangered, this could be catastrophic for the future of the language.    

# 2. Methodology

TonER Framework definition:

First we run the translations through the models and then we take the reference text (which is the correct translation) and the model output. To prevent the entire evaluation from breaking if the AI hallucinates or deletes an entire word, we use a *word-level alignment* strategy. What we do is:

1. Split the reference sentence and the model output into individual words.
2. Strip all the marks from these words to create plain, "base" words.
3. Align the base words from the reference to the base words from the model output.
4. For the words that match, we zoom in and compare them character-by-character (graphemes), including their original marks.

We also "normalize" the text which ensures the marks are all the same format in the computer. We classify each reference grapheme into one of three categories (tonal-only, orthographic-only, or mixed).

TonER works by checking positions where a mark *should* exist according to the human reference. Then, if the model deletes or incorrectly substitutes a required mark, it counts as an error. If the model hallucinates a new mark on a plain letter, or hallucinates an entirely unaligned word, it is ignored. This specifically isolates the AI's ability to "remember" to use expected diacritics.

We created 2 new variations of a common metric (DER) in this study to evaluate the accuracy of the model:

**1. Character-Level Diacritic Error Rate (CDER)**
This measures the exact proportion of individual marks the model failed to restore. "c" represents a "category" or an individual language.

$$\text{CDER}_c = \frac{\sum \text{errors}_c}{\sum \text{total}_c} $$

*Simply put: The number of "mark" errors divided by the total possible "marks" for each category. For example, a 0.5 CDER means the AI got 50% of the individual marks wrong.*

**2. Word-Level Diacritic Error Rate (WDER)**
This measures the proportion of *words* containing a specific mark type that had at least one error of that type.

$$\text{WDER}_c = \frac{\sum \text{words with error}_c}{\sum \text{total words with mark}_c} $$

*Simply put: The number of times a word was wrong because of a mark being wrong divided by the total possible words the model could have gotten right.*

We also use 2 common metrics to get a general sense of the model's accuracy for comparison:

**1. BLEU (Bilingual Evaluation Understudy)**
This measures how many exact words and phrases the model's translation shares with the original correct reference.

**2. chrF++ (Character n-gram F-score)**
This evaluates the overlap of character sequences and word sequences between the translation and the reference. It is similar to BLEU, but it gives the AI partial credit for getting parts of a word right, making it a fairer test for languages with complex spelling.

Ultimately, we compute Character-Level and Word-Level Error Rates for Tonal (TDER/WDER_T), Orthographic (ODER/WDER_O), and Mixed (MDER/WDER_M) categories as well as BLEU and chrF++ for each model and language pair.

For this experimental setup, we evaluate two models that are commonly used for this task:

(These are the same base model but they have different parameters which means one is larger and was trained on more data which presumably makes it more accurate)

1. NLLB-200-distilled-600M
2. NLLB-200-distilled-1.3B

(Costa-jussà et al., 2022)

on the FLORES+ benchmark (a dataset made by Meta to evaluate these models) for Ewe, Igbo, and Yorùbá.

In [ ]:
!pip install sacrebleu

In [ ]:
# citations:
# https://huggingface.co/docs/huggingface_hub/quick-start
import huggingface_hub
from google.colab import userdata

# Retrieve the Hugging Face token from Colab secrets and strip any whitespace
HF_TOKEN = userdata.get('HF_TOKEN').strip()

# Log in to Hugging Face using the retrieved token to access the gated dataset
huggingface_hub.login(token=HF_TOKEN)

In [ ]:
# citations:
# https://colab.research.google.com/notebooks/io.ipynb

from google.colab import drive

# Connect to Google Drive so we can safely save our final evaluation results later
drive.mount('/content/drive')

# Data Ingestion and Orthographic Filtering

To ensure our evaluation of diacritical marks is accurate, we must strictly control the quality of the text we feed into the program. This step downloads the test datasets for English, Ewe, Igbo, and Yorùbá (Goyal et al., 2022). Because computers can hide the same character under different digital codes, we immediately apply a process called Unicode Normalization (Davis et al., 2001) to guarantee every letter is represented in the exact same format. We then define the official, standardized alphabets for each target language so the computer knows exactly what to look for. By filtering out any sentences containing foreign characters or typos, we prevent bad data from confusing the AI or skewing our error rates. This careful cleaning ensures that when the model makes a mistake, it is a genuine failure of the AI's understanding, not just a problem with a messy dataset.

In [ ]:
# citations:
# https://huggingface.co/docs/datasets/loading
# https://docs.python.org/3/library/unicodedata.html
# https://pypi.org/project/regex/

from datasets import load_dataset
import unicodedata
import torch
import regex
import difflib
import gc
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Load the benchmark data
# We use FLORES+, a high-quality Meta dataset for translation.
english_d = load_dataset("openlanguagedata/flores_plus", "eng_Latn")
ewe_d = load_dataset("openlanguagedata/flores_plus", "ewe_Latn")
igbo_d = load_dataset("openlanguagedata/flores_plus", "ibo_Latn")
yoruba_d = load_dataset("openlanguagedata/flores_plus", "yor_Latn")

# 2. Normalize Text
# Forces every character into a standard Unicode format (NFC) so computers can compare them perfectly.
def normalize_dataset(example):
    example['text'] = unicodedata.normalize('NFC', example['text'])
    return example

english_n = english_d.map(normalize_dataset)
ewe_n = ewe_d.map(normalize_dataset)
igbo_n = igbo_d.map(normalize_dataset)
yoruba_n = yoruba_d.map(normalize_dataset)

# 3. Define the Official Alphabets
# Here we define which characters count as 'tonal' vs 'orthographic'.

# Yorùbá: Bamgbose (1965)
yoruba_tonal_str = "áàéèíìóòúùẹ́ẹ̀ọ́ọ̀ńǹḿm̀"
yoruba_orth_str = "ẹọṣẹ́ẹ̀ọ́ọ̀"
yoruba_total_str = "abcdefghijklmnopqrstuwxyẹọṣáàéèíìóòúùẹ́ẹ̀ọ́ọ̀ńǹḿm̀"

# Igbo: Onwu Committee (1961)
igbo_tonal_str = "áàéèíìóòúùị́ị̀ọ́ọ̀ụ́ụ̀ńǹḿm̀"
igbo_orth_str = "ịọụṅị́ị̀ọ́ọ̀ụ́ụ̀"
igbo_total_str = "abcdefghijklmnopqrstuvwxyzịọụṅáàéèíìóòúùị́ị̀ọ́ọ̀ụ́ụ̀ńǹḿm̀"

# Ewe: Capo (1991) / Gbe Orthography
ewe_tonal_str = "áàǎâéèěêɛ́ɛ̀ɛ̌ɛ̂íìǐîóòǒôɔ́ɔ̀ɔ̌ɔ̂úùǔû"
ewe_orth_str = "ɖɛƒɣŋɔʋɛ́ɛ̀ɛ̌ɛ̂ɔ́ɔ̀ɔ̌ɔ̂"
ewe_total_str = "abcdefghijklmnopqrstuvwxyzɖɛƒɣŋɔʋáàǎâéèěêɛ́ɛ̀ɛ̌ɛ̂íìǐîóòǒôɔ́ɔ̀ɔ̌ɔ̂úùǔû"

# Helper function to break a string down into unique, normalized characters (graphemes)
def make_grapheme_set(text):
    normalized_text = unicodedata.normalize('NFC', text)
    return set(regex.findall(r'\X', normalized_text))

yoruba_valid = make_grapheme_set(yoruba_total_str)
igbo_valid = make_grapheme_set(igbo_total_str)
ewe_valid = make_grapheme_set(ewe_total_str)

# Group the sets dynamically by language to replace the old messy global if/else blocks
SETS = {
    "yor_Latn": {
        "T": make_grapheme_set(yoruba_tonal_str),
        "O": make_grapheme_set(yoruba_orth_str),
        "valid": yoruba_valid
    },
    "ibo_Latn": {
        "T": make_grapheme_set(igbo_tonal_str),
        "O": make_grapheme_set(igbo_orth_str),
        "valid": igbo_valid
    },
    "ewe_Latn": {
        "T": make_grapheme_set(ewe_tonal_str),
        "O": make_grapheme_set(ewe_orth_str),
        "valid": ewe_valid
    }
}

# 4. Filter the Datasets
# We throw away sentences with foreign characters to ensure our error counts are accurate.
def filter_standard_orthography(example, valid_graphemes):
    text_lower = unicodedata.normalize('NFC', example['text'].lower())
    for g in regex.findall(r'\X', text_lower):
        if any(c.islower() for c in g) and g not in valid_graphemes:
            return False
    return True

# Apply the language-specific filters
yoruba_f = yoruba_n.filter(lambda x: filter_standard_orthography(x, yoruba_valid))
igbo_f = igbo_n.filter(lambda x: filter_standard_orthography(x, igbo_valid))
ewe_f = ewe_n.filter(lambda x: filter_standard_orthography(x, ewe_valid))

# Ensure the English datasets match the surviving sentences exactly
def get_ids(dataset_split):
    return set(dataset_split['id'])

english_ewe_f = {}
english_igbo_f = {}
english_yoruba_f = {}

for split in ['dev', 'devtest']:
    ewe_ids = get_ids(ewe_f[split])
    igbo_ids = get_ids(igbo_f[split])
    yoruba_ids = get_ids(yoruba_f[split])

    english_ewe_f[split] = english_n[split].filter(lambda x: x['id'] in ewe_ids)
    english_igbo_f[split] = english_n[split].filter(lambda x: x['id'] in igbo_ids)
    english_yoruba_f[split] = english_n[split].filter(lambda x: x['id'] in yoruba_ids)

# Print pure sentence counts
print("ewe:\n", len(ewe_f['devtest']) + len(ewe_f['dev']), "\n")
print("igbo:\n", len(igbo_f['devtest']) + len(igbo_f['dev']), "\n")
print("yoruba:\n", len(yoruba_f['devtest']) + len(yoruba_f['dev']), "\n")

# Translation Generation and Error Alignment

This section implements the core mechanics of the TonER framework, which is how we actually measure the AI's mistakes. It provides routines to translate English text into the target languages using the AI models we downloaded. We run these translations in "batches" on a graphics card (GPU) because translating one sentence at a time on a standard computer processor would take far too long. Following translation, we use a sequence-matching algorithm (Ratcliff & Obershelp, 1988), like an automated spot-the-difference tool. It lines up the AI's output characters directly against the perfect human translation. The algorithm meticulously steps through the text letter-by-letter to classify every missing or incorrect mark as a tonal (pitch), orthographic (base letter), or mixed error. This allows us to isolate exactly where the AI is failing instead of just giving it a single, unhelpful overall score.

In [ ]:
# citations:
# https://huggingface.co/docs/transformers/main_classes/text_generation
# https://huggingface.co/docs/transformers/multilingual#nllb
# https://docs.python.org/3/library/difflib.html

def translate_batch(texts, model, tokenizer, tgt_lang_code, model_name, batch_size=16):
    # Translates a batch of English sentences simultaneously on the GPU for speed.
    all_outputs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        tokenizer.src_lang = "eng_Latn"

        # Convert words to numerical IDs for the AI
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(device)

        # Force the model to start speaking in the target language
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang_code)

        # Generate the translation
        with torch.no_grad():
            output = model.generate(**inputs, forced_bos_token_id=forced_bos_token_id,
                                    max_new_tokens=256, num_beams=4, early_stopping=True)

        # Convert numbers back to text
        for ids in output:
            decoded = tokenizer.decode(ids, skip_special_tokens=True)
            all_outputs.append(unicodedata.normalize("NFC", decoded))

    return all_outputs

CLASSES = ('T', 'O', 'M')

def get_grapheme_type(g_nfc, lang_sets):
    # Checks the specific language dictionary to see if the mark is Tonal, Ortho, or Mixed.
    has_tone = g_nfc in lang_sets["T"]
    has_ortho = g_nfc in lang_sets["O"]
    if has_tone and has_ortho: return "M"
    if has_tone: return "T"
    if has_ortho: return "O"
    return None

def strip_grapheme(g):
    # Remove all combining diacritics -> base letter(s), lowercase.
    return ''.join(
        c for c in unicodedata.normalize('NFD', g)
        if unicodedata.category(c) != 'Mn'
    ).lower()

def strip_token(tok):
    # Strips all marks from a full word to create a 'base' word for alignment.
    return ''.join(strip_grapheme(g) for g in regex.findall(r'\X', unicodedata.normalize('NFC', tok)))

def count_diacritic_errors(reference, model_output, lang_sets):
    # Step 1: Split sentences into words and normalize
    ref_toks = regex.findall(r'\b\w+\b', unicodedata.normalize('NFC', reference.lower()))
    hyp_toks = regex.findall(r'\b\w+\b', unicodedata.normalize('NFC', model_output.lower()))

    # Step 2: Align words by their stripped (undiacritised) forms.
    rs = [strip_token(t) for t in ref_toks]
    hs = [strip_token(t) for t in hyp_toks]
    m  = difflib.SequenceMatcher(None, rs, hs, autojunk=False)

    aligned_pairs = [
        (ref_toks[i], hyp_toks[j])
        for tag, i1, i2, j1, j2 in m.get_opcodes()
        if tag == 'equal'
        for i, j in zip(range(i1, i2), range(j1, j2))
    ]

    # Initialize counts for this sentence
    counts = {c: {"total": 0, "err": 0, "del": 0, "sub": 0, "words_total": 0, "words_err": 0} for c in CLASSES}

    # Step 3: Iterate through the matched words and compare characters (including marks)
    for w_ref, w_hyp in aligned_pairs:
        rg = regex.findall(r'\X', w_ref)
        hg = regex.findall(r'\X', w_hyp)

        word_has_mark = {c: False for c in CLASSES}
        word_has_err = {c: False for c in CLASSES}

        for i, gr in enumerate(rg):
            cls = get_grapheme_type(gr, lang_sets)
            if cls is None:
                continue

            counts[cls]["total"] += 1
            word_has_mark[cls] = True

            gh = hg[i] if i < len(hg) else None
            # If the output character doesn't perfectly match the reference (case-insensitive)
            if gh is None or unicodedata.normalize('NFC', gr.lower()) != unicodedata.normalize('NFC', gh.lower()):
                counts[cls]["err"] += 1
                word_has_err[cls] = True

                # If the AI provided a valid character of any type here but it was wrong, it's a substitution.
                # If the AI dropped the character entirely or provided a plain letter, it's a deletion.
                if gh and get_grapheme_type(gh, lang_sets) is not None:
                    counts[cls]["sub"] += 1
                else:
                    counts[cls]["del"] += 1

        for c in CLASSES:
            if word_has_mark[c]:
                counts[c]["words_total"] += 1
            if word_has_err[c]:
                counts[c]["words_err"] += 1

    return counts

# Metric Aggregation and Evaluation Orchestration

With the character-level logic established, this pipeline organizes the evaluation over the entire cleaned dataset. It pairs the English source texts with their perfect human references and processes them through the translation functions we just built. Instead of just looking at one sentence, it accumulates the exact counts of errors versus the total possible places a mark *could* have been right across thousands of sentences. From these grand totals, it calculates the overarching Diacritic Error Rates—specifically, the Tonal Error Rate (TDER), Orthographic Error Rate (ODER), and Mixed Error Rate (MDER). This systematic counting ensures our final statistical metrics reflect the AI model's true performance on a massive scale. By grouping the errors this way, we can prove mathematically (e.g., via Wilcoxon signed-rank tests; Wilcoxon, 1945) whether the model struggles more with spoken tone marks than written letter marks.

In [ ]:
# citations:
# https://github.com/mjpost/sacrebleu

from tqdm.auto import tqdm
import pandas as pd
import unicodedata
import sacrebleu
# Sacrebleu is the industry standard for calculating BLEU and chrF++ translation metrics


# Metrics setup
_bleu = sacrebleu.metrics.BLEU(effective_order=True)
_chrf = sacrebleu.metrics.CHRF(word_order=2)

def evaluate_translation_dataset(
    english_dataset,
    target_dataset,
    model,
    tokenizer,
    tgt_lang_code,
    model_name,
    max_samples=None,
    batch_size=16
):
    # Master coordinator function for a single language translation and evaluation.
    results = []
    n = max_samples if max_samples else len(target_dataset)

    eng_texts = [unicodedata.normalize('NFC', english_dataset[i]['text']) for i in range(min(n, len(english_dataset)))]
    tgt_texts = [unicodedata.normalize('NFC', target_dataset[i]['text']) for i in range(min(n, len(target_dataset)))]

    print(f"Translating en→{tgt_lang_code} (Batched)")
    pred_texts = translate_batch(eng_texts, model, tokenizer, tgt_lang_code, model_name, batch_size=batch_size)

    # Get the specific alphabet rules for this language
    lang_sets = SETS[tgt_lang_code]

    # Setup aggregation dictionaries for grand totals
    agg = {c: {"total": 0, "err": 0, "del": 0, "sub": 0, "words_total": 0, "words_err": 0} for c in CLASSES}

    # 2. Go sentence-by-sentence and evaluate
    for eng_text, ref_text, pred_text in zip(eng_texts, tgt_texts, pred_texts):
        counts = count_diacritic_errors(ref_text, pred_text, lang_sets)

        for cat in CLASSES:
            for k in counts[cat]:
                agg[cat][k] += counts[cat][k]

        # Save detailed breakdown for this sentence
        results.append({
            'english_source': eng_text,
            'gold_reference': ref_text,
            'model_output': pred_text
        })

    def rate(num, denom):
        return round(num / denom, 4) if denom > 0 else 0

    # Package all our findings up neatly into a summary report
    summary = {
        'language': tgt_lang_code,
        "tder": rate(agg["T"]["err"], agg["T"]["total"]),
        "oder": rate(agg["O"]["err"], agg["O"]["total"]),
        "mder": rate(agg["M"]["err"], agg["M"]["total"]),
        "WDER_T": rate(agg["T"]["words_err"], agg["T"]["words_total"]),
        "WDER_O": rate(agg["O"]["words_err"], agg["O"]["words_total"]),
        "WDER_M": rate(agg["M"]["words_err"], agg["M"]["words_total"]),
        "total_tonal_graphemes": agg["T"]["total"],
        "total_ortho_graphemes": agg["O"]["total"],
        "total_tonal_deletions": agg["T"]["del"],
        "total_tonal_substitutions": agg["T"]["sub"],
        "total_ortho_deletions": agg["O"]["del"],
        "total_ortho_substitutions": agg["O"]["sub"],
        "BLEU": round(_bleu.corpus_score(pred_texts, [tgt_texts]).score, 2),
        "chrF++": round(_chrf.corpus_score(pred_texts, [tgt_texts]).score, 2),
    }

    return summary, pd.DataFrame(results)

# Experiment Execution and Result Archival

The final component manages the massive execution loop across multiple AI translation models and languages. Because these AI models are incredibly large and require massive amounts of computer memory, we dynamically load them into the graphics card using a "half-precision" trick that shrinks their size without losing much accuracy. It systematically evaluates each language, and to prevent the computer from crashing from full memory, it explicitly deletes the old model before loading the next one. Upon completion, the script aggregates the comprehensive statistical summaries and sentence-by-sentence evaluation data. Finally, it exports them as structured files to the Google Drive we connected earlier. These saved files act as the concrete proof for our research, allowing us to deeply analyze the AI's behavior later without having to rerun the hours-long translation process.

In [ ]:
# citations:
# https://huggingface.co/docs/transformers/perf_infer_gpu_one
# https://pytorch.org/docs/stable/notes/cuda.html#memory-management

from datasets import concatenate_datasets
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import gc
import pandas as pd
import os
from datetime import datetime

# 1. Set up our test subjects
language_configs = {
    "Ewe": {
        "dataset": concatenate_datasets([ewe_f['dev'], ewe_f['devtest']]),
        "english_dataset": concatenate_datasets([english_ewe_f['dev'], english_ewe_f['devtest']]),
        "lang_code": "ewe_Latn"
    },
    "Igbo": {
        "dataset": concatenate_datasets([igbo_f['dev'], igbo_f['devtest']]),
        "english_dataset": concatenate_datasets([english_igbo_f['dev'], english_igbo_f['devtest']]),
        "lang_code": "ibo_Latn"
    },
    "Yoruba": {
        "dataset": concatenate_datasets([yoruba_f['dev'], yoruba_f['devtest']]),
        "english_dataset": concatenate_datasets([english_yoruba_f['dev'], english_yoruba_f['devtest']]),
        "lang_code": "yor_Latn"
    }
}

model_configs = [
    "facebook/nllb-200-distilled-600M",
    "facebook/nllb-200-distilled-1.3B",
]

all_summaries = []
all_results_dfs = {}
# Use fast graphics card (GPU) if available
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Main Experiment Loop
for model_name in model_configs:
    print(f"\nloading Model: {model_name}")

    # Download the model and place it into the GPU using half-precision (float16)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        dtype=torch.float16,
        device_map="auto",
        tie_word_embeddings=False # Silences the Hugging Face warning about untied weights
    )
    model.eval() # Sets the model to test mode

    all_results_dfs[model_name] = {}

    # Loop over every language
    for lang, config in language_configs.items():
        print(f"\n evaluating {lang} with {model_name}")
        summary, results_df = evaluate_translation_dataset(
            english_dataset=config['english_dataset'],
            target_dataset=config['dataset'],
            model=model,
            tokenizer=tokenizer,
            tgt_lang_code=config['lang_code'],
            model_name=model_name,
            max_samples=None
        )

        summary['model'] = model_name
        all_summaries.append(summary)
        all_results_dfs[model_name][lang] = results_df

    # Memory Cleanup
    # AI models are huge. We MUST delete the old one from GPU memory before loading the next.
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 3. Display & Save Results
summary_df = pd.DataFrame(all_summaries)
display(summary_df)

# Create a timestamped folder in Google Drive
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"/content/drive/MyDrive/toner_results_{timestamp}"
os.makedirs(output_dir, exist_ok=True)

# Save the high-level summary tables
summary_df.to_csv(f"{output_dir}/summary.csv", index=False)
summary_df.to_parquet(f"{output_dir}/summary.parquet")

# Save the highly detailed sentence-by-sentence data (parquet is much faster/smaller than json/csv)
for model_name, lang_dfs in all_results_dfs.items():
    safe_model = model_name.replace('/', '_')
    for lang, df in lang_dfs.items():
        df.to_parquet(f"{output_dir}/sentences_{safe_model}_{lang}.parquet")

### Visualizing the TonER Findings
In this cell, we create two figures to represent the data. The following two charts were chosen because they illustrate the two core arguments of the paper: the performance gap between mark types, and the "blanking" failure mode. The first chart is a bar graph displaying the DER rates for both models and all three languages. The second figure is another bar graph that displays what the actual error breakdown was for each error type for each model. We also do significance testing with a Chi-square analysis which is a statistical method used to determine if there is a significant relationship between categorical variables. Instead of testing sentence-by-sentence, we pool all characters across the entire dataset. This tests if the overall proportion of tonal errors is significantly different from the overall proportion of orthographic errors.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

# Set visual style
sns.set_theme(style="whitegrid")

# FIGURE 1: TDER vs ODER (The Performance Gap & Scaling)
# Reshape data to plot CDER and WDER side-by-side easily
df_melted = summary_df.melt(id_vars=['language', 'model'],
                            value_vars=['tder', 'oder', 'WDER_T', 'WDER_O'],
                            var_name='Metric',
                            value_name='Error Rate')

# Clean up the labels for the chart
df_melted['Metric'] = df_melted['Metric'].map({
    'tder': 'TDER (Char Tonal)',
    'oder': 'ODER (Char Ortho)',
    'WDER_T': 'WDER (Word Tonal)',
    'WDER_O': 'WDER (Word Ortho)'
})

# Create a side-by-side bar chart, split by model
g = sns.catplot(
    data=df_melted, kind='bar',
    x='language', y='Error Rate', hue='Metric', col='model',
    height=5, aspect=1.5, palette="muted"
)
g.fig.suptitle('Figure 1: Tonal vs. Orthographic Error Rates (Character & Word Level)', y=1.05)
plt.show()

# FIGURE 2: Tonal Deletions vs Substitutions
plt.figure(figsize=(8, 5))

# Sum the error types across all languages for each model
error_behaviors = summary_df.groupby('model')[['total_tonal_deletions', 'total_tonal_substitutions']].sum().reset_index()

# Reshape for seaborn
behaviors_melted = error_behaviors.melt(id_vars='model',
                                        value_vars=['total_tonal_deletions', 'total_tonal_substitutions'],
                                        var_name='Error Type',
                                        value_name='Total Count')

behaviors_melted['Error Type'] = behaviors_melted['Error Type'].map({
    'total_tonal_deletions': 'Deletions (Omissions)',
    'total_tonal_substitutions': 'Substitutions (Incorrect Marks)'
})

# Create the bar chart
sns.barplot(data=behaviors_melted, x='model', y='Total Count', hue='Error Type', palette="pastel")

plt.title('Figure 2: Tonal Error Behaviors (Deletions vs. Substitutions')
plt.ylabel('Total Count of Errors')
plt.xlabel('Model')
plt.tight_layout()
plt.show()


print("Character-Level (CDER) Significance Testing:\n")
for index, row in summary_df.iterrows():
    # Reconstruct the exact raw error counts directly from the precise percentages
    # (Using the recall logic: Total expected marks vs Errors)
    tonal_err = int(round(row['tder'] * row['total_tonal_graphemes']))
    ortho_err = int(round(row['oder'] * row['total_ortho_graphemes']))

    tonal_correct = int(row['total_tonal_graphemes']) - tonal_err
    ortho_correct = int(row['total_ortho_graphemes']) - ortho_err

    # Create a 2x2 table: [Errors, Correct (Total - Errors)]
    contingency_table = [
        [tonal_err, tonal_correct],   # Tonal Bucket
        [ortho_err, ortho_correct]    # Ortho Bucket
    ]

    # Run the Chi-Square test
    stat, p_value, dof, expected = chi2_contingency(contingency_table)

    is_sig = "SIGNIFICANT" if p_value < 0.05 else "NOT significant"
    print(f"{row['model']} ({row['language']}): p-value = {p_value:.5e} ({is_sig})")

# 3. Results

Across both NLLB-200 variants, Igbo and Yorùbá show substantial tonal-orthographic gaps. For the 600M parameter model evaluating Yorùbá, TDER (tonal error rate) sits at 0.4589 compared to an ODER (orthographic error rate) of 0.1047. The pattern becomes even more interesting when making the model bigger: shifting from 600M to 1.3B parameters actually *improves* orthographic restoration for Yorùbá (ODER drops to 0.0666) but causes tonal restoration to get worse (TDER increases to 0.7347). We use a chi-square test to confirm that the difference is statistically significant (p = 0.0).

Word-level metrics also demonstrate that these tonal errors affect overall sentence meaning rather than being isolated to a few problematic words. The WDER_T (Word-level Tonal Error Rate) for Yorùbá on the 1.3B model is 0.7343. This means that over 73% of all translated words that are supposed to contain a tone mark fail to restore it correctly, fundamentally corrupting the meaning of the vast majority of the text.


# 4. Discussion

Based on our data, TonER seems to detect critical imbalances that traditional metrics hide (Zitouni et al., 2006). For example, a single overall Diacritic Error Rate for the NLLB-200-1.3B model on Yorùbá would output 40% failure rate but mask the 73% tonal error rate and the 6% orthographic error rate. (Costa-jussà et al., 2022). By separating these mark types, TonER reveals that the model's translation quality on these languages is heavily bottlenecked specifically by tonal restoration.

Furthermore, tonal errors are overwhelmingly deletions rather than substitutions. In the Yorùbá 1.3B evaluation, out of 12,030 total expected tonal graphemes, the model committed 8,632 deletions (omitting the tone mark entirely to produce a plain letter) and only 207 substitutions (producing the wrong mark). This confirms a distinct "blanking" effect where the model defaults to the base character rather than making incorrect guesses.

We suggest this hypothesis:
The massive scale of these errors and the deletion-dominated failure mode could be a phenomenon known as frequency-driven representational collapse (Gao et al., 2019).

Because tone marks are routinely omitted by native speakers in casual online text, which serves as the primary data source for training models, the internal mathematical representation of these marks blurs or "collapses" (Bernas et al., 2026). This means that the model learns that tone marks are statistically rare and "unimportant," and instead defaults to the plain, unmarked base letters.

This is heavily supported by the scaling observation. We saw that as the model increased from 600M to 1.3B parameters, it developed worse tonal errors. In the phenomenon we suggest, this is due to the model being more mathematically confident that unmarked letters are the norm in its training data. This makes the larger and "smarter" model suppress the rare tone marks even more aggressively than the smaller model, opting to blank them out (over 8,600 deletions) rather than risk an incorrect guess (Martinez et al., 2024).

# 5. Limitations

While the TonER framework successfully isolates error modalities, this study has several limitations. First, restrictive orthographic filtering, while necessary to ensure data purity (Goyal et al., 2022), can severely limit the sample size for certain languages. For instance, the Ewe evaluation set yielded only 75 valid tonal graphemes compared to Yorùbá's 12,030, limiting the statistical resolution of the Ewe findings despite the chi-square significance.

Second, the framework relies on automatic word-level alignment using a standard sequence-matching algorithm (Ratcliff & Obershelp, 1988). If a neural model hallucinates entirely unrelated text or produces a translation so poor that base words cannot be reliably matched to the human reference, those words are dropped from the evaluation. While TonER's strict recall basis (only evaluating confidently aligned words) prevents false error inflation, it may inadvertently undercount errors in the most catastrophically translated sentences.

Finally, this evaluation is limited to the NLLB-200 encoder-decoder family (Costa-jussà et al., 2022). Modern decoder-only Large Language Models (LLMs) trained with different objectives and tokenization schemes may exhibit different diacritic behavior patterns that TonER has yet to test (Touvron et al., 2023).

# 6. Future work

First, we must expand the current evaluation to a wider variety of architectures, such as Google's MADLAD 3B/7B (Kudugunta et al., 2023) and modern decoder-only LLMs (Touvron et al., 2023), to determine if this tonal-orthographic gap is a universal property of neural machine translation or an artifact of NLLB's specific training regimen.

Second, to mathematically prove the "representational collapse" theory proposed here, we need a mechanistic verification pipeline. This would involve directly computing the geometric dispersion—measuring the distance between the AI's internal mathematical vectors—of tonal versus orthographic tokens in the model's high-dimensional embedding space (Li et al., 2026). If tonal embeddings are demonstrably clustered closer to their unmarked base counterparts than orthographic tokens are, it would provide hard evidence of representational blurring.

Finally, we should explore "representation surgery"—manually adjusting the AI's hidden numbers while it generates text (Hernandez et al., 2024). By artificially boosting the activation vectors associated with tonal marks during the generation phase, we could potentially force the model to overcome its frequency bias and restore marks without needing to retrain the multi-million dollar model from scratch. Resolving this internal blurriness has been shown to improve translation quality globally (Tokarchuk et al., 2026); TonER provides the necessary framework to ask if this approach can save low-resource African languages.

# 7. Conclusion

This paper asked whether there is a measurable difference in the error rates of different types of diacritical marks in neural machine translation. The answer is yes, and the difference is large. Across Igbo and Yorùbá, tonal diacritic error rates substantially exceeded orthographic rates at both the character and word level, and this gap was consistent across both model sizes.

The more striking finding is that scaling made it worse, not better. As the model grew from 600M to 1.3B parameters, orthographic restoration improved while tonal restoration degraded further. This confirms that the two mark types are failing for different reasons, and that any evaluation metric combining them into a single number is obscuring the most important part of the result.

TonER exists to make that distinction visible. By separating tonal and orthographic error rates, it gives researchers a precise view of where translation models are failing on diacritic-heavy languages and a framework for directing improvements accordingly. For Yorùbá, Igbo, and Ewe, where tonal marks encode meaning that cannot be recovered from context alone, this level of specificity matters. Standard metrics that average over mark types do not just underreport errors — they make it harder to fix them.

---

# 8. Bibliography

*   Adelani, D. I., et al. (2022). A few thousand translations go a long way! Leveraging pre-trained models for African news translation.
*   Bamgbose, A. (1965). *Yoruba orthography*. Ibadan University Press.
*   Bernas, et al. (2026). Anisotropy in low-resource language representations. *Journal of Machine Learning Research*.
*   Capo, H. B. C. (1991). *A comparative phonology of Gbe*. Foris Publications.
*   Costa-jussà, M. R., et al. (2022). No language left behind: Scaling human-centered machine translation.
*   Davis, M., et al. (2001). *Unicode standard annex #15: Unicode normalization forms*.
*   Ezeani, I., et al. (2017). Igbo diacritic restoration using embedding models.
*   Gao, J., et al. (2019). Representation degeneration problem in training natural language generation models.
*   Goyal, N., et al. (2022). The FLORES-101 evaluation benchmark for low-resource and multilingual machine translation.
*   Hernandez, E., et al. (2024). Linearity of relation decoding in transformer language models.
*   Kudugunta, S., et al. (2023). MADLAD-400: A multilingual and document-level large audited dataset.
*   Li, et al. (2026). Geometric dispersion of tokens in multilingual LLMs.
*   Martinez, et al. (2024). Frequency-driven anisotropy in decoder models.
*   Nekoto, W., et al. (2020). Participatory research for low-resourced machine translation: A case study in African languages.
*   Onwu Committee. (1961). *The official Igbo orthography*.
*   Orife, I. (2018). Attentive sequence-to-sequence learning for diacritic restoration of Yorùbá language text.
*   Ratcliff, J. W., & Obershelp, D. (1988). Pattern matching: The gestalt approach.
*   Tokarchuk, et al. (2026). Angular dispersion regularization for NMT.
*   Touvron, H., et al. (2023). LLaMA: Open and Efficient Foundation Language Models. *arXiv preprint arXiv:2302.13971*.
*   Wilcoxon, F. (1945). Individual comparisons by ranking methods.
*   Zitouni, I., et al. (2006). Maximum entropy based restoration of Arabic diacritics.